## Example 1: Basic HTTP GET Request

In [ ]:
# Builtin retry for flaky external endpoints — retries transient network errors with backoff.
import time as _t
def _retry(_fn, _tries=5, _delay=1.0):
    for _i in range(_tries):
        try:
            return _fn()
        except Exception:
            if _i == _tries - 1: raise
            _t.sleep(_delay * (_i + 1))
import pycurl
import io
import json

# Example 1: Basic HTTP GET Request
print("Example 1 - Basic HTTP GET Request")

# Create a Curl object
curl = pycurl.Curl()

# Set the URL
test_url = "https://httpbingo.org/get"
curl.setopt(curl.URL, test_url)

# Create a buffer to store the response
response_buffer = io.BytesIO()
curl.setopt(curl.WRITEDATA, response_buffer)

# Set timeout and user agent
curl.setopt(curl.TIMEOUT, 30)
curl.setopt(curl.USERAGENT, "pycurl-test/1.0")

print(f"Making GET request to: {test_url}")

# Perform the request
_retry(lambda: curl.perform())

# Get response information
http_code = curl.getinfo(curl.RESPONSE_CODE)
total_time = curl.getinfo(curl.TOTAL_TIME)

print(f"HTTP Status: {http_code}")
print(f"Total Time: {total_time:.2f}s")

# Get the response body
response_body = response_buffer.getvalue().decode('utf-8')
response_data = json.loads(response_body)

print(f"Response URL: {response_data.get('url')}")
print(f"Headers count: {len(response_data.get('headers', {}))}")

# Clean up
curl.close()

print("\nExample 1 completed successfully")

## Example 2: HTTP POST request with data and headers

In [ ]:
import pycurl
import io
import json

# Example 2: HTTP POST request with data and headers
print("pycurl module imported successfully")

try:
    print("Example 2 - HTTP POST request with data and headers")
    
    # Create a Curl object
    curl = pycurl.Curl()
    
    # Set the URL
    post_url = "https://httpbingo.org/post"
    curl.setopt(curl.URL, post_url)
    
    # Set HTTP method to POST
    curl.setopt(curl.POST, 1)
    
    # Prepare POST data
    post_data = {
        "username": "testuser",
        "email": "test@example.com",
        "message": "Hello from pycurl!",
        "timestamp": "2024-01-01T12:00:00Z"
    }
    
    # Convert to JSON
    json_data = json.dumps(post_data)
    curl.setopt(curl.POSTFIELDS, json_data)
    
    # Set headers
    headers = [
        "Content-Type: application/json",
        "Accept: application/json",
        "User-Agent: pycurl-test-client/1.0",
        "X-Test-Header: test-value"
    ]
    curl.setopt(curl.HTTPHEADER, headers)
    
    # Create buffers for response and headers
    response_buffer = io.BytesIO()
    header_buffer = io.BytesIO()
    
    curl.setopt(curl.WRITEDATA, response_buffer)
    curl.setopt(curl.HEADERFUNCTION, header_buffer.write)
    
    # Set timeout and other options
    curl.setopt(curl.TIMEOUT, 30)
    curl.setopt(curl.FOLLOWLOCATION, 1)
    curl.setopt(curl.SSL_VERIFYPEER, 1)
    
    print(f"Making POST request to: {post_url}")
    print(f"POST data: {post_data}")
    
    # Perform the request
    _retry(lambda: curl.perform())
    
    # Get response information
    http_code = curl.getinfo(curl.RESPONSE_CODE)
    total_time = curl.getinfo(curl.TOTAL_TIME)
    upload_speed = curl.getinfo(curl.SPEED_UPLOAD)
    content_length = curl.getinfo(curl.CONTENT_LENGTH_DOWNLOAD)
    
    print(f"HTTP Status Code: {http_code}")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Upload Speed: {upload_speed:.0f} bytes/sec")
    print(f"Content Length: {content_length}")
    
    # Parse response
    response_body = response_buffer.getvalue().decode('utf-8')
    response_data = json.loads(response_body)
    
    # Show sent data
    print(f"\nSent data verification:")
    sent_data = json.loads(response_data.get('data', '{}'))
    print(f"  Username: {sent_data.get('username', 'N/A')}")
    print(f"  Email: {sent_data.get('email', 'N/A')}")
    print(f"  Message: {sent_data.get('message', 'N/A')}")
    
    # Show headers received by server
    received_headers = response_data.get('headers', {})
    print(f"\nHeaders received by server:")
    header_keys = ['Content-Type', 'User-Agent', 'X-Test-Header', 'Content-Length']
    for header_key in header_keys:
        if header_key in received_headers:
            print(f"  {header_key}: {received_headers[header_key]}")
    
    # Parse response headers
    response_headers = header_buffer.getvalue().decode('utf-8')
    header_lines = [line.strip() for line in response_headers.split('\n') if line.strip()]
    
    print(f"\nResponse headers:")
    status_line = header_lines[0] if header_lines else "N/A"
    print(f"  Status: {status_line}")
    
    # Show content type and server headers
    content_headers = [line for line in header_lines if 'content-type' in line.lower() or 'server' in line.lower()]
    for header in content_headers[:3]:  # Show first 3 relevant headers
        print(f"  {header}")
    
    # Show form data if present
    form_data = response_data.get('form', {})
    if form_data:
        print(f"\nForm data received: {form_data}")
    
    # Show JSON data received
    json_received = response_data.get('json', {})
    if json_received:
        print(f"JSON data received: {len(json_received)} fields")
    
    # Clean up
    curl.close()
    
    print("HTTP POST request with data and headers completed successfully")
    print("pycurl example 2 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycurl example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 2 finished")

## Example 3: File upload and progress tracking

In [ ]:
import pycurl
import io
import json

# Example 3: File upload and progress tracking
print("pycurl module imported successfully")

try:
    print("Example 3 - File upload and progress tracking")
    
    # Create some test content to upload
    test_content = """This is a test file for pycurl upload demonstration.
It contains multiple lines of text to simulate a real file upload.
Line 3: Testing multipart form data upload.
Line 4: Demonstrating progress tracking capabilities.
Line 5: End of test content."""
    
    # Create a file-like object from the content
    file_buffer = io.BytesIO(test_content.encode('utf-8'))
    file_size = len(test_content.encode('utf-8'))
    
    print(f"Created test content ({file_size} bytes)")
    
    # Progress tracking function
    def progress_callback(download_total, downloaded, upload_total, uploaded):
        if upload_total > 0:
            upload_percent = (uploaded / upload_total) * 100
            print(f"  Upload progress: {upload_percent:.1f}% ({uploaded}/{upload_total} bytes)")
        return 0  # Return 0 to continue
    
    # Create a Curl object
    curl = pycurl.Curl()
    
    # Set the URL for file upload
    upload_url = "https://httpbingo.org/post"
    curl.setopt(curl.URL, upload_url)
    
    # Configure multipart form data upload
    curl.setopt(curl.POST, 1)
    
    # Create multipart form data
    form_data = [
        ('file', (curl.FORM_BUFFER, 'test.txt', curl.FORM_BUFFERPTR, test_content.encode('utf-8'))),
        ('description', 'Test file upload via pycurl'),
        ('category', 'test'),
        ('timestamp', '2024-01-01T12:00:00Z')
    ]
    
    curl.setopt(curl.HTTPPOST, form_data)
    
    # Set up progress tracking
    curl.setopt(curl.PROGRESSFUNCTION, progress_callback)
    curl.setopt(curl.NOPROGRESS, 0)  # Enable progress reporting
    
    # Set headers
    headers = [
        "User-Agent: pycurl-upload-client/1.0",
        "Accept: application/json"
    ]
    curl.setopt(curl.HTTPHEADER, headers)
    
    # Create buffers for response
    response_buffer = io.BytesIO()
    header_buffer = io.BytesIO()
    
    curl.setopt(curl.WRITEDATA, response_buffer)
    curl.setopt(curl.HEADERFUNCTION, header_buffer.write)
    
    # Set timeout and SSL options
    curl.setopt(curl.TIMEOUT, 60)
    curl.setopt(curl.SSL_VERIFYPEER, 1)
    curl.setopt(curl.FOLLOWLOCATION, 1)
    
    print(f"Starting multipart file upload to: {upload_url}")
    
    # Perform the upload
    curl.perform()
    
    # Get upload information
    http_code = curl.getinfo(curl.RESPONSE_CODE)
    total_time = curl.getinfo(curl.TOTAL_TIME)
    upload_size = curl.getinfo(curl.SIZE_UPLOAD)
    upload_speed = curl.getinfo(curl.SPEED_UPLOAD)
    
    print(f"\nUpload completed!")
    print(f"HTTP Status Code: {http_code}")
    print(f"Total Time: {total_time:.2f}s")
    print(f"Upload Size: {upload_size} bytes")
    print(f"Upload Speed: {upload_speed:.0f} bytes/sec")
    
    # Parse response
    response_body = response_buffer.getvalue().decode('utf-8')
    response_data = json.loads(response_body)
    
    # Show uploaded files
    files_data = response_data.get('files', {})
    form_data_received = response_data.get('form', {})
    
    print(f"\nUploaded files:")
    for filename, content in files_data.items():
        content_preview = content[:100] + "..." if len(content) > 100 else content
        print(f"  {filename}: {len(content)} bytes")
        print(f"    Preview: {content_preview}")
    
    print(f"\nForm data received:")
    for field, value in form_data_received.items():
        print(f"  {field}: {value}")
    
    # Show request headers
    request_headers = response_data.get('headers', {})
    print(f"\nRequest headers:")
    interesting_headers = ['Content-Type', 'Content-Length', 'User-Agent']
    for header in interesting_headers:
        if header in request_headers:
            value = request_headers[header]
            # Truncate very long headers
            if len(value) > 100:
                value = value[:100] + "..."
            print(f"  {header}: {value}")
    
    # Check for multipart boundary
    content_type = request_headers.get('Content-Type', '')
    if 'multipart/form-data' in content_type:
        print(f"  ✓ Multipart form data upload successful")
        if 'boundary=' in content_type:
            boundary_part = content_type.split('boundary=')[1][:20]
            print(f"  Boundary: {boundary_part}...")
    
    # Show timing information
    namelookup_time = curl.getinfo(curl.NAMELOOKUP_TIME)
    connect_time = curl.getinfo(curl.CONNECT_TIME)
    pretransfer_time = curl.getinfo(curl.PRETRANSFER_TIME)
    
    print(f"\nTiming breakdown:")
    print(f"  DNS lookup: {namelookup_time:.3f}s")
    print(f"  Connect: {connect_time:.3f}s")
    print(f"  Pre-transfer: {pretransfer_time:.3f}s")
    print(f"  Transfer: {total_time - pretransfer_time:.3f}s")
    
    # Clean up
    curl.close()
    
    print("File upload and progress tracking completed successfully")
    print("pycurl example 3 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycurl example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 3 finished")

## Example 4: Multiple concurrent requests with CurlMulti

In [ ]:
import pycurl
import io

# Example 4: Multiple concurrent requests with CurlMulti
print("pycurl module imported successfully")

try:
    print("Example 4 - Multiple concurrent requests with CurlMulti")
    
    # URLs to fetch concurrently
    test_urls = [
        "https://httpbingo.org/delay/1",  # 1 second delay
        "https://httpbingo.org/delay/2",  # 2 second delay  
        "https://httpbingo.org/status/200",  # Immediate response
        "https://httpbingo.org/json",      # JSON response
        "https://httpbingo.org/user-agent" # User agent echo
    ]
    
    print(f"Setting up {len(test_urls)} concurrent requests")
    
    # Create CurlMulti object for handling multiple requests
    multi = pycurl.CurlMulti()
    
    # Storage for curl objects and their response buffers
    curl_objects = []
    response_buffers = []
    
    # Create individual Curl objects for each URL
    for i, url in enumerate(test_urls):
        curl = pycurl.Curl()
        response_buffer = io.BytesIO()
        
        # Configure the curl object
        curl.setopt(curl.URL, url)
        curl.setopt(curl.WRITEDATA, response_buffer)
        curl.setopt(curl.TIMEOUT, 30)
        curl.setopt(curl.USERAGENT, f"pycurl-multi-client-{i}/1.0")
        curl.setopt(curl.SSL_VERIFYPEER, 1)
        curl.setopt(curl.FOLLOWLOCATION, 1)
        
        # Add to multi handle
        multi.add_handle(curl)
        
        # Store references
        curl_objects.append(curl)
        response_buffers.append(response_buffer)
        
        print(f"  Request {i+1}: {url}")
    
    print(f"\nStarting {len(curl_objects)} concurrent requests...")
    
    # Perform all requests concurrently
    import time
    start_time = time.time()
    
    # Process requests
    while True:
        # Perform non-blocking operations
        ret, active_handles = multi.perform()
        
        if active_handles == 0:
            break
            
        # Wait for activity on file descriptors
        multi.select(1.0)
    
    end_time = time.time()
    total_time = end_time - start_time
    
    print(f"All requests completed in {total_time:.2f} seconds")
    
    # Process responses
    print(f"\nProcessing responses:")
    
    for i, (curl, buffer) in enumerate(zip(curl_objects, response_buffers)):
        url = test_urls[i]
        
        # Get response information
        http_code = curl.getinfo(curl.RESPONSE_CODE)
        response_time = curl.getinfo(curl.TOTAL_TIME)
        download_size = curl.getinfo(curl.SIZE_DOWNLOAD)
        
        print(f"\n  Request {i+1}: {url}")
        print(f"    Status: {http_code}")
        print(f"    Time: {response_time:.2f}s")
        print(f"    Size: {download_size} bytes")
        
        # Get response content
        response_content = buffer.getvalue().decode('utf-8', errors='ignore')
        
        # Show response preview based on URL type
        if '/json' in url:
            try:
                import json
                json_data = json.loads(response_content)
                print(f"    JSON keys: {list(json_data.keys())}")
            except:
                print(f"    Content preview: {response_content[:100]}...")
                raise
        elif '/user-agent' in url:
            try:
                import json
                json_data = json.loads(response_content)
                user_agent = json_data.get('user-agent', 'N/A')
                print(f"    User-Agent: {user_agent}")
            except:
                print(f"    Content preview: {response_content[:100]}...")
                raise
        elif '/delay/' in url:
            try:
                import json
                json_data = json.loads(response_content)
                args = json_data.get('args', {})
                print(f"    Delay response args: {args}")
            except:
                print(f"    Content preview: {response_content[:100]}...")
                raise
        else:
            # Show first 100 characters of response
            preview = response_content[:100].replace('\n', '\\n')
            print(f"    Content preview: {preview}...")
    
    # Show timing analysis
    print(f"\nTiming analysis:")
    individual_times = [curl.getinfo(curl.TOTAL_TIME) for curl in curl_objects]
    sequential_time = sum(individual_times)
    time_saved = sequential_time - total_time
    efficiency = (time_saved / sequential_time) * 100 if sequential_time > 0 else 0
    
    print(f"  Total concurrent time: {total_time:.2f}s")
    print(f"  Sequential time would be: {sequential_time:.2f}s")
    print(f"  Time saved: {time_saved:.2f}s ({efficiency:.1f}% efficiency gain)")
    
    # Show individual request timing
    print(f"  Individual request times:")
    for i, req_time in enumerate(individual_times):
        print(f"    Request {i+1}: {req_time:.2f}s")
    
    # Get overall multi statistics
    print(f"\nMulti handle statistics:")
    print(f"  Requests processed: {len(curl_objects)}")
    print(f"  Active handles at start: {len(curl_objects)}")
    print(f"  Active handles at end: 0")
    
    # Performance metrics
    total_downloaded = sum(curl.getinfo(curl.SIZE_DOWNLOAD) for curl in curl_objects)
    avg_speed = total_downloaded / total_time if total_time > 0 else 0
    
    print(f"  Total downloaded: {total_downloaded} bytes")
    print(f"  Average speed: {avg_speed:.0f} bytes/sec")
    
    # Clean up
    for curl in curl_objects:
        multi.remove_handle(curl)
        curl.close()
    multi.close()
    
    print("Multiple concurrent requests completed successfully")
    print("pycurl example 4 completed successfully")
    
except ImportError as e:
    print(f"Import error: {e}")
except Exception as e:
    print(f"Runtime error: {e}")
    raise AssertionError(
        f"pycurl example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e
    
print("Example 4 finished")

## Example 5: Authentication and Custom Methods

In [ ]:
import pycurl
import io
import json

# Example 5: Authentication and Custom Methods
print("Example 5 - Authentication and Custom Methods")

# Test basic authentication
curl = pycurl.Curl()
auth_url = "https://httpbingo.org/basic-auth/testuser/testpass"
curl.setopt(curl.URL, auth_url)

# Set basic authentication
curl.setopt(curl.USERPWD, "testuser:testpass")
curl.setopt(curl.HTTPAUTH, curl.HTTPAUTH_BASIC)

response_buffer = io.BytesIO()
curl.setopt(curl.WRITEDATA, response_buffer)

print(f"Testing basic auth at: {auth_url}")
curl.perform()

http_code = curl.getinfo(curl.RESPONSE_CODE)
print(f"Auth status: {http_code}")

if http_code == 200:
    response_data = json.loads(response_buffer.getvalue().decode('utf-8'))
    print(f"Authenticated user: {response_data.get('user')}")
    print("Authentication successful ✓")

# Test custom HTTP method (PUT)
curl.setopt(curl.URL, "https://httpbingo.org/put")
curl.setopt(curl.CUSTOMREQUEST, "PUT")

put_data = json.dumps({"method": "PUT", "data": "test"})
curl.setopt(curl.POSTFIELDS, put_data)

response_buffer = io.BytesIO()
curl.setopt(curl.WRITEDATA, response_buffer)

curl.perform()
put_status = curl.getinfo(curl.RESPONSE_CODE)
print(f"\nPUT request status: {put_status}")

curl.close()

print("\nExample 5 completed successfully")